In [15]:
import sys
from pathlib import Path
import pandas as pd
import fitz  
import tiktoken
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))


from system.rag import (
    LLM,
    Approaches,
    RAGExperimentRunner,
    CSVProcessor,
    LangfairMetricsCalculator,
    LangfairRunner,
    ApproachRetrievers,
)

from system.preprocess import (
    PDFPreprocessConfig,
    PDFPreprocessor,
    CorpusBuilderConfig,
    CorpusBuilder,
)


from system.evaluation import (
    JudgeBatchConfig,
    JudgeBatchBuilder,
    BatchResultsConfig,
    BatchResultsExporter,
    JudgeMergeConfig,
    JudgeResultsMerger,
    CSVColumnMergeConfig, 
    CSVColumnMerger
)

# Check if Tokens will fit in the context window

## CROP PDFS

In [16]:
PDF_DIR = Path("../data/input/input_pdfs")

TARGETS = {
    "Mill (Bridgeport)": {
        "path": PDF_DIR / "Bridgeport Series 1 Milling manual with schematics.pdf",
        "crop_top": 0.04, "crop_bottom": 0.075, "crop_left": 0.0, "crop_right": 0.0,
    },
    "UR5e Cobot": {
        "path": PDF_DIR / "UR5e_Universal_Robots User Manual.pdf",
        "crop_percent": 0.075,
    },
}

In [17]:
from langchain_community.document_loaders import PyMuPDFLoader

def load_pdf_as_text(path) -> str:
    pages = PyMuPDFLoader(str(path)).load()
    return "\n\n".join(p.page_content for p in pages).strip()

mill_text = load_pdf_as_text(TARGETS["Mill (Bridgeport)"]["path"])
ur5e_text = load_pdf_as_text(TARGETS["UR5e Cobot"]["path"])

print(f"Mill  : {len(mill_text):,} chars")
print(f"UR5e  : {len(ur5e_text):,} chars")

Mill  : 123,442 chars
UR5e  : 276,138 chars


In [18]:
doc = fitz.open(TARGETS["UR5e Cobot"]["path"])

### Crop PDFs and extract text

In [19]:
OUTPUT_DIR = Path("../data/input/cropped_pdfs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CROPPED = {}
for label, cfg in TARGETS.items():
    out = OUTPUT_DIR / f"cropped_{cfg['path'].name}"
    PDFPreprocessor.crop_pdf(
        cfg["path"], out,
        crop_percent=cfg.get("crop_percent", 0.075),
        crop_top=cfg.get("crop_top"),
        crop_bottom=cfg.get("crop_bottom"),
        crop_left=cfg.get("crop_left"),
        crop_right=cfg.get("crop_right"),
    )
    CROPPED[label] = out
    print(f"Cropped {label} -> {out.name}")

Cropped Mill (Bridgeport) -> cropped_Bridgeport Series 1 Milling manual with schematics.pdf
Cropped UR5e Cobot -> cropped_UR5e_Universal_Robots User Manual.pdf


In [20]:
mill_cropped_text = load_pdf_as_text(CROPPED["Mill (Bridgeport)"])
ur5e_cropped_text = load_pdf_as_text(CROPPED["UR5e Cobot"])

print(f"Mill  (cropped): {len(mill_cropped_text):,} chars")
print(f"UR5e  (cropped): {len(ur5e_cropped_text):,} chars")

Mill  (cropped): 121,502 chars
UR5e  (cropped): 253,244 chars


In [21]:
enc = tiktoken.get_encoding("cl100k_base")

for label, cfg in TARGETS.items():
    orig = load_pdf_as_text(cfg["path"])
    cropped = load_pdf_as_text(CROPPED[label])
    orig_tokens = len(enc.encode(orig))
    crop_tokens = len(enc.encode(cropped))
    print(f"{label}")
    print(f"  Original : {orig_tokens:,} tokens")
    print(f"  Cropped  : {crop_tokens:,} tokens")
    print(f"  Diff     : {orig_tokens - crop_tokens:,} tokens removed")
    print()

Mill (Bridgeport)
  Original : 46,606 tokens
  Cropped  : 45,319 tokens
  Diff     : 1,287 tokens removed

UR5e Cobot
  Original : 68,416 tokens
  Cropped  : 60,754 tokens
  Diff     : 7,662 tokens removed



# Long-Context RAG Evaluation

In [ ]:
from system.utils import EnvironmentConfig, read_text

env_config = EnvironmentConfig()
rets = ApproachRetrievers(env_config)
rets.set_long_context_texts({
    "Mill (Bridgeport)": mill_cropped_text,
    "UR5e Cobot": ur5e_cropped_text,
})

## Run Experiment — Mill

In [ ]:
runner = RAGExperimentRunner(
    retrievers=rets,
    num_replicates=1,
    approaches=Approaches.LONG_CONTEXT,
    models=LLM.GPT_5_MINI_2025_08_07 | LLM.GPT_5_NANO_2025_08_07,
    max_tokens_list=[5000],
    efforts=["low"],
    topk_list=[1],
    ans_instr_A=read_text("../data/prompts/ans_instr_A.txt"),
    fewshot_A=read_text("../data/prompts/fewshot_A.txt"),
    max_concurrent=5,
    max_chars_per_content=500_000,
)

MILL_QA = Path("../data/QA/MILL/Mill Feedback Accepted.csv")
MILL_OUTPUT = Path("../data/results/RAG_Output/LongContext/MILL_LONG_CONTEXT_OUTPUT.csv")
await runner.run(MILL_QA, MILL_OUTPUT)

## Run Experiment — UR5e

In [ ]:
UR5E_QA = Path("../data/QA/UR5E/UR5e_QA.csv")  # create this file with question,gold_answer columns
UR5E_OUTPUT = Path("../data/results/RAG_Output/LongContext/UR5E_LONG_CONTEXT_OUTPUT.csv")
await runner.run(UR5E_QA, UR5E_OUTPUT)

# Compute Similarity Metrics

In [ ]:
metrics_runner = LangfairRunner(
    calculator=LangfairMetricsCalculator(),
    processor=CSVProcessor(),
    max_concurrent=500,
)
await metrics_runner.run(q_a_csv=MILL_OUTPUT, out_csv=None)
# await metrics_runner.run(q_a_csv=UR5E_OUTPUT, out_csv=None)

# Judge Batch (Helpfulness & Correctness)

In [ ]:
judge_config = JudgeBatchConfig(
    csv_path=MILL_OUTPUT,
    output_jsonl=Path("../data/results/batchprocess/MILL_LONG_CONTEXT_BATCH.jsonl"),
    judge_model="gpt-5",
    completion_window="24h",
    submit_to_openai=False,  # set True when ready
    env_file=None,
)
builder = JudgeBatchBuilder(judge_config)
result = builder.run()
print(f"Prepared {result['num_requests']} requests (submitted={result['submitted']})")

# Parse Judge Results
After batch completes, update `batch_id` below and uncomment.

In [ ]:
# csv_config = BatchResultsConfig(
#     batch_id="batch_xxx",  # replace with real batch ID
#     raw_jsonl_path=Path("../data/results/batchprocess/mill_lc_batch_raw.jsonl"),
#     json_output_path=Path("../data/results/batchprocess/mill_lc_batch.json"),
#     csv_output_path=Path("../data/results/batchprocess/mill_lc_batch.csv"),
# )
# BatchResultsExporter().download_records(csv_config)
#
# merge_config = JudgeMergeConfig(
#     input_csv=csv_config.csv_output_path,
#     output_csv=Path("../data/results/batchprocess/mill_lc_batch_wide.csv"),
# )
# JudgeResultsMerger().run(merge_config)
#
# final_merge = CSVColumnMergeConfig(
#     left_csv=Path(str(MILL_OUTPUT).replace(".csv", "_with_metrics.csv")),
#     right_csv=merge_config.output_csv,
#     output_csv=Path("../data/results/final_merged/mill_long_context.csv"),
#     on="permutation_id",
#     how="left",
#     exclude_columns=["q", "retrieved_files", "meta_hits_text"],
# )
# CSVColumnMerger().run(final_merge)

# Analysis

In [ ]:
from system.analysis import analyze_csv

analyze_csv(
    csv_input=MILL_OUTPUT,
    output_dir=Path("../data/results/final_merged/analysis_mill_long_context/"),
)
# analyze_csv(
#     csv_input=UR5E_OUTPUT,
#     output_dir=Path("../data/results/final_merged/analysis_ur5e_long_context/"),
# )